# **This notebook performs an additional analysis on validation set and runs inference on model trained with domain adversarial learning**

# **Import libraries**

In [1]:
# Standard Library Imports
import os              # File and directory operations
import math            # Mathematical functions
import shutil          # High-level file operations (copy, move, delete)
import csv             # CSV file reading/writing
from collections import defaultdict

# Data Handling & Processing
import numpy as np         # Numerical computations and arrays
import pandas as pd        # Data manipulation and analysis
import h5py                # HDF5 file handling

# Progress & Visualization
from tqdm import tqdm      # Progress bars for loops
import matplotlib.pyplot as plt  # Plotting and visualization

# PyTorch: Deep Learning
import torch                         # Core PyTorch library
import torch.nn as nn                # Neural network modules
import torch.optim as optim          # Optimizers (Adam, AdamW, etc.)
from torch.utils.data import (       # Dataset and DataLoader utilities
    DataLoader,
    Dataset
)


# **Model architecture**


In [2]:
# Gradient Reversal Layer (GRL)
# Used for domain-adversarial training
class GradReverse(torch.autograd.Function):
    """
    Gradient Reversal Layer (GRL) reverses gradients during backprop.
    This encourages the feature extractor to learn domain-invariant features.
    """
    @staticmethod
    def forward(ctx, x, lambd):
        """
        Forward pass: identity operation.

        Parameters
        ----------
        x : torch.Tensor
            Input features.
        lambd : float
            Gradient reversal coefficient.

        Returns
        -------
        x : torch.Tensor
            Same as input (identity).
        """
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        """
        Backward pass: reverse the gradient and scale by lambda.
        """
        return grad_output.neg() * ctx.lambd, None

def grad_reverse(x, lambd):
    """Convenience function to apply GRL."""
    return GradReverse.apply(x, lambd)

# Token Embedding Module
# Converts raw EEG signals into learned embeddings for LSTM
class TokenEmbedding(nn.Module):
    """
    Convolutional token embedding for EEG channels and temporal sequence.
    """
    def __init__(self, c_in, d_model):
        """
        Parameters
        ----------
        c_in : int
            Number of EEG input channels.
        d_model : int
            Embedding dimension for each time step.
        """
        super(TokenEmbedding, self).__init__()

        # Temporal convolution: expands channels to d_model*4
        self.embed_layer = nn.Sequential(
            nn.Conv2d(1, d_model * 4, kernel_size=(1, 8), padding='same'),
            nn.BatchNorm2d(d_model * 4),
            nn.GELU()
        )

        # Spatial convolution across channels: reduces to d_model
        self.embed_layer2 = nn.Sequential(
            nn.Conv2d(d_model * 4, d_model, kernel_size=(c_in, 1), padding='valid'),
            nn.BatchNorm2d(d_model),
            nn.GELU()
        )

    def forward(self, x):
        """
        Forward pass of token embedding.

        Parameters
        ----------
        x : torch.Tensor
            Input EEG tensor of shape (B, C, T)

        Returns
        -------
        x : torch.Tensor
            Embedded tokens of shape (B, T, d_model)
        """
        x = x.unsqueeze(1)            # Add channel dimension: (B, 1, C, T)
        x = self.embed_layer(x)       # Temporal conv
        x = self.embed_layer2(x)      # Spatial conv across channels
        x = x.squeeze(2)              # Remove singleton channel dim
        x = x.permute(0, 2, 1)        # (B, T, d_model) for LSTM
        return x

# DARNet + LSTM (Standard Model)
class DARNet_LSTM(nn.Module):
    """
    DARNet with LSTM for EEG classification (task only, no domain adaptation).
    """
    def __init__(self, c_in=32, d_model=16, hidden=64, num_classes=2):
        super().__init__()
        self.token_embed = TokenEmbedding(c_in, d_model)
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(hidden*2, num_classes)  # bidirectional

    def forward(self, x):
        """
        Forward pass.

        Parameters
        ----------
        x : torch.Tensor
            EEG batch (B, C, T)

        Returns
        -------
        out : torch.Tensor
            Task logits (B, num_classes)
        """
        x = self.token_embed(x)        # Token embeddings: (B, T, d_model)
        x, _ = self.lstm(x)            # LSTM output: (B, T, hidden*2)
        x = x.mean(dim=1)              # Global average pooling over time
        out = self.classifier(x)       # Task logits
        return out


# DARNet + LSTM + DANN (Domain-Adversarial)
class DARNet_LSTM_DANN(nn.Module):
    """
    DARNet with LSTM + Domain-Adversarial Neural Network (DANN)
    - Task classifier predicts auditory attention
    - Domain classifier predicts subject ID with GRL
    """
    def __init__(self, c_in=32, d_model=16, hidden=64, num_classes=2, num_subjects=30):
        super().__init__()

        # Token embedding module
        self.token_embed = TokenEmbedding(c_in, d_model)

        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Task classifier (main task)
        self.classifier = nn.Linear(hidden*2, num_classes)

        # Domain classifier (subject prediction)
        self.domain_classifier = nn.Sequential(
            nn.Linear(hidden*2, 64),
            nn.ReLU(),
            nn.Linear(64, num_subjects)
        )

    def forward(self, x, lambd=0.0):
        """
        Forward pass with optional gradient reversal.

        Parameters
        ----------
        x : torch.Tensor
            EEG batch (B, C, T)
        lambd : float
            Gradient reversal factor (0 during evaluation)

        Returns
        -------
        logits_task : torch.Tensor
            Task prediction logits (B, num_classes)
        logits_domain : torch.Tensor
            Domain (subject) prediction logits (B, num_subjects)
        """
        emb = self.token_embed(x)                  # Token embeddings
        features, _ = self.lstm(emb)              # LSTM output
        feat = features.mean(dim=1)               # Global average pooling

        # Task prediction
        logits_task = self.classifier(feat)

        # Domain prediction with gradient reversal
        rev = grad_reverse(feat, lambd)
        logits_domain = self.domain_classifier(rev)

        return logits_task, logits_domain


# **Load the model checkpoints**

In [3]:
seed_paths = {
    42: "/kaggle/input/datasets/sumanpunshi123/eeg-aad-task1/best_model_seed42/best_model_seed42",
    100: "/kaggle/input/datasets/sumanpunshi123/eeg-aad-task1/best_model_seed100/best_model_seed100",
    0: "/kaggle/input/datasets/sumanpunshi123/eeg-aad-task1/best_model_seed0/best_model_seed0",
    1: "/kaggle/input/datasets/sumanpunshi123/eeg-aad-task1/best_model_seed1/best_model_seed1",
}

# **Calculating extra metric of subject std on validation set and subject level confidence intervals**

## **Function to load data from h5 file**

In [4]:
def load_h5_dataset_val(file_path):
    """
    Load EEG validation data, labels, and subject IDs from an HDF5 (.h5) file.

    Parameters
    ----------
    file_path : str
        Path to the HDF5 file containing 'data', 'label', and 'sub_id' datasets.

    Returns
    -------
    data : np.ndarray
        EEG data array of shape (N, channels, time_points), e.g., (N, 32, 128).
    label : np.ndarray
        Corresponding labels for each sample.
    subjects : np.ndarray
        1D array of subject IDs associated with each sample.
    """
    # Open the HDF5 file in read-only mode to prevent accidental modifications
    with h5py.File(file_path, 'r') as f:
        data = np.array(f['data'])
        label = np.array(f['label'])
        subjects = np.array(f['sub_id'])
    # Squeeze subject array to remove extra dimension
    return data, label, subjects.squeeze()


## **Define custom class for validation data**

In [5]:
class CustomDataset(Dataset):
    """
    PyTorch Dataset for EEG data with labels and subject IDs.

    This class allows seamless integration with PyTorch DataLoader,
    providing batches of (data, label, subject) tensors for training
    and evaluation, including domain-adversarial setups.
    """

    def __init__(self, data, labels, subjects):
        """
        Initialize the dataset.

        Parameters
        ----------
        data : array-like or np.ndarray
            Input EEG data of shape (N, channels, time_points).
        labels : array-like or np.ndarray
            Integer labels for each sample (task targets).
        subjects : array-like or np.ndarray
            Subject IDs corresponding to each sample (used for domain adaptation).
        """
        self.data = data
        self.labels = labels
        self.subjects = subjects

    def __len__(self):
        """
        Return the total number of samples in the dataset.
        """
        return len(self.labels)

    def __getitem__(self, index):
        """
        Retrieve a single sample by index.

        Returns
        -------
        x : torch.FloatTensor
            EEG data for the given sample.
        y : torch.LongTensor
            Label for the given sample.
        s : torch.LongTensor
            Subject ID for the given sample.
        """
        # Convert NumPy arrays to PyTorch tensors
        x = torch.tensor(self.data[index], dtype=torch.float32)
        y = torch.tensor(self.labels[index], dtype=torch.long)
        s = torch.tensor(self.subjects[index], dtype=torch.long)

        return x, y, s


## **Function to evaluate the data**

In [6]:
def evaluate_model(model, val_loader, device="cuda"):

    model.eval()
    total_correct = 0
    total_samples = 0
    subject_correct = defaultdict(int)
    subject_total = defaultdict(int)
    with torch.no_grad():
        for x, y, s in tqdm(val_loader, desc="Evaluating"):
            x = x.to(device)
            y = y.to(device).long().squeeze(-1)
            s = s.to(device)
            # Task prediction
            logits, _ = model(x)
            preds = torch.argmax(logits, dim=1)
            # Overall accuracy
            total_correct += (preds == y).sum().item()
            total_samples += y.size(0)
            # Subject-level accuracy
            for yi, pi, si in zip(
                y.cpu(),
                preds.cpu(),
                s.cpu()
            ):
                sid = int(si)
                subject_total[sid] += 1
                if yi == pi:
                    subject_correct[sid] += 1

    # Overall accuracy
    overall_accuracy = total_correct / total_samples

    # Subject-level accuracies
    subject_accuracies = {
        sid: subject_correct[sid] / subject_total[sid]
        for sid in subject_total
    }

    return overall_accuracy, subject_accuracies

## **Function for subject level confidence interval**

In [7]:
def bootstrap_subject_ci(
    subject_mean_accs,
    n_bootstrap=10000,
    seed=42
):

    rng = np.random.default_rng(seed)

    subject_accs = np.array(
        list(subject_mean_accs.values())
    )

    observed_mean = np.mean(subject_accs)

    bootstrap_means = np.empty(n_bootstrap)

    for i in range(n_bootstrap):

        # Resample SUBJECTS, not EEG windows
        sampled = rng.choice(
            subject_accs,
            size=len(subject_accs),
            replace=True
        )

        bootstrap_means[i] = np.mean(sampled)

    lower = np.percentile(
        bootstrap_means,
        2.5
    )

    upper = np.percentile(
        bootstrap_means,
        97.5
    )

    return observed_mean, lower, upper


## **Load the validation set**

In [8]:
val_h5_path = "/kaggle/input/datasets/sumanpunshi123/eeg-aad-task1/val_data.h5"

X_val, y_val, s_val = load_h5_dataset_val(val_h5_path)

# Convert subject IDs to 0-based indexing
unique_subjects = np.unique(s_val)
subject2idx = {sub: i for i, sub in enumerate(unique_subjects)}
s_val = np.array([subject2idx[s] for s in s_val])

val_loader = DataLoader(
    CustomDataset(X_val, y_val, s_val),
    batch_size=128,
    shuffle=False
)

device = "cuda"


## **Evaluate all four seeds**

In [9]:
import os
import zipfile
import torch

all_seed_overall_acc = {}
all_seed_subject_accs = {}

for seed, checkpoint_path in seed_paths.items():

    print("\n" + "=" * 60)
    print(f"Evaluating model seed: {seed}")
    print("=" * 60)

    # 1. FIXED FOR KAGGLE DIRECTORIES (WITH CORRECT INTERNAL SUBDIRECTORY STRUCTURE):
    temp_zip_file = f"/kaggle/working/reconstructed_model_seed_{seed}.pt"
    
    print(f"Bypassing Kaggle directory structure for seed {seed}...")
    with zipfile.ZipFile(temp_zip_file, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(checkpoint_path):
            for file in files:
                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(full_path, checkpoint_path)
                
                # CRITICAL FIX: PyTorch demands files live inside an internal subfolder wrapper.
                # We prefix everything with 'archive/' to appease inline_container.cc
                internal_zip_path = os.path.join("archive", rel_path)
                
                zipf.write(full_path, internal_zip_path)
                
    print(f"Successfully packed folder to file: {temp_zip_file}")

    # 2. Initialize model
    model = DARNet_LSTM_DANN(
        d_model=8,
        num_subjects=30
    )

    # 3. Load checkpoint from the structured file
    state_dict = torch.load(
        temp_zip_file,
        map_location=device
    )

    model.load_state_dict(state_dict)
    model.to(device)

    # Evaluate
    overall_acc, subject_accs = evaluate_model(
        model,
        val_loader,
        device=device
    )

    # Store results
    all_seed_overall_acc[seed] = overall_acc
    all_seed_subject_accs[seed] = subject_accs

    # Print results
    print(f"\nOverall accuracy: {overall_acc * 100:.2f}%")

    print("\nSubject-level accuracies:")
    for subject, acc in subject_accs.items():
        print(f"  Subject {subject}: {acc * 100:.2f}%")

    # Clean up the temporary file to save disk space
    if os.path.exists(temp_zip_file):
        os.remove(temp_zip_file)



Evaluating model seed: 42
Bypassing Kaggle directory structure for seed 42...
Successfully packed folder to file: /kaggle/working/reconstructed_model_seed_42.pt


Evaluating:   0%|          | 0/206 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv2d(
Evaluating: 100%|██████████| 206/206 [00:08<00:00, 24.35it/s]



Overall accuracy: 53.03%

Subject-level accuracies:
  Subject 0: 51.11%
  Subject 1: 52.96%
  Subject 2: 50.64%
  Subject 3: 57.42%

Evaluating model seed: 100
Bypassing Kaggle directory structure for seed 100...
Successfully packed folder to file: /kaggle/working/reconstructed_model_seed_100.pt


Evaluating: 100%|██████████| 206/206 [00:07<00:00, 27.19it/s]



Overall accuracy: 52.44%

Subject-level accuracies:
  Subject 0: 47.99%
  Subject 1: 47.54%
  Subject 2: 54.64%
  Subject 3: 59.59%

Evaluating model seed: 0
Bypassing Kaggle directory structure for seed 0...
Successfully packed folder to file: /kaggle/working/reconstructed_model_seed_0.pt


Evaluating: 100%|██████████| 206/206 [00:07<00:00, 27.23it/s]



Overall accuracy: 52.94%

Subject-level accuracies:
  Subject 0: 52.16%
  Subject 1: 45.08%
  Subject 2: 52.80%
  Subject 3: 61.72%

Evaluating model seed: 1
Bypassing Kaggle directory structure for seed 1...
Successfully packed folder to file: /kaggle/working/reconstructed_model_seed_1.pt


Evaluating: 100%|██████████| 206/206 [00:07<00:00, 27.05it/s]


Overall accuracy: 53.50%

Subject-level accuracies:
  Subject 0: 47.46%
  Subject 1: 58.01%
  Subject 2: 52.22%
  Subject 3: 56.29%


## **Overall Accuracy accross seeds**

In [10]:
seed_accuracies = np.array([
    acc for acc in all_seed_overall_acc.values()
])

mean_accuracy = np.mean(seed_accuracies)

std_accuracy = np.std(
    seed_accuracies,
    ddof=1
)

print("\n" + "=" * 60)
print("OVERALL PERFORMANCE ACROSS MODEL SEEDS")
print("=" * 60)

for seed, acc in all_seed_overall_acc.items():
    print(f"Seed {seed}: {acc * 100:.2f}%")

print(
    f"\nValidation Accuracy: "
    f"{mean_accuracy * 100:.2f} ± "
    f"{std_accuracy * 100:.2f}%"
)


OVERALL PERFORMANCE ACROSS MODEL SEEDS
Seed 42: 53.03%
Seed 100: 52.44%
Seed 0: 52.94%
Seed 1: 53.50%

Validation Accuracy: 52.98 ± 0.43%


## **Subject level accuracy across seeds and confidence interval**

In [11]:
common_subjects = sorted(
    set.intersection(
        *[
            set(subject_accs.keys())
            for subject_accs in all_seed_subject_accs.values()
        ]
    )
)

subject_mean_accs = {}

for subject in common_subjects:

    accuracies = np.array([
        all_seed_subject_accs[seed][subject]
        for seed in all_seed_subject_accs
    ])

    subject_mean_accs[subject] = np.mean(accuracies)


print("\n" + "=" * 60)
print("SUBJECT-LEVEL PERFORMANCE")
print("=" * 60)

for subject, acc in subject_mean_accs.items():
    print(
        f"Subject {subject}: "
        f"{acc * 100:.2f}%"
    )

subject_accuracy_values = list(subject_mean_accs.values())
std_sub= np.std(subject_accuracy_values)   
print(f"Population STD: {std_sub}")


subject_mean, ci_lower, ci_upper = bootstrap_subject_ci(
    subject_mean_accs
)


print("\n" + "=" * 60)
print("SUBJECT-LEVEL CONFIDENCE INTERVAL")
print("=" * 60)

print(
    f"Mean subject-level accuracy: "
    f"{subject_mean * 100:.2f}%"
)

print(
    f"95% CI: "
    f"[{ci_lower * 100:.2f}%, "
    f"{ci_upper * 100:.2f}%]"
)


SUBJECT-LEVEL PERFORMANCE
Subject 0: 49.68%
Subject 1: 50.90%
Subject 2: 52.57%
Subject 3: 58.75%
Population STD: 0.03490261640510711

SUBJECT-LEVEL CONFIDENCE INTERVAL
Mean subject-level accuracy: 52.98%
95% CI: [50.29%, 56.79%]


## **Result display**

In [12]:
print("\n")
print("=" * 70)
print("FINAL VALIDATION RESULTS")
print("=" * 70)

print("\nAccuracy across four model seeds:")

for seed, acc in all_seed_overall_acc.items():
    print(
        f"  Seed {seed}: "
        f"{acc * 100:.2f}%"
    )

print(
    f"\nMean ± SD across seeds: "
    f"{mean_accuracy * 100:.2f} ± "
    f"{std_accuracy * 100:.2f}%"
)

print(
    f"\nMean subject-level accuracy: "
    f"{subject_mean * 100:.2f}%"
)

print(
    f"95% subject-level CI: "
    f"[{ci_lower * 100:.2f}%, "
    f"{ci_upper * 100:.2f}%]"
)

print("=" * 70)



FINAL VALIDATION RESULTS

Accuracy across four model seeds:
  Seed 42: 53.03%
  Seed 100: 52.44%
  Seed 0: 52.94%
  Seed 1: 53.50%

Mean ± SD across seeds: 52.98 ± 0.43%

Mean subject-level accuracy: 52.98%
95% subject-level CI: [50.29%, 56.79%]
